# Travel Assistant Agent — Using LangGraph, Vector DB, External APIs

An end-to-end travel assistant project that builds a LangGraph agent with tools for flight search (Amadeus), hotel search (Amadeus + Tavily fallback), restaurant search (Tavily), weather lookup (Tavily), currency exchange (Tavily), FAQ retrieval, and trip planning using a **LanceDB** vector database of an FAQ set and a parsed travel book. Includes a CLI interface and a comprehensive scenario-testing suite.

## Import All Libraries

In [ ]:
import os
import re
import uuid
import time
import json
import difflib
import lancedb
import kagglehub
import pandas as pd
from dotenv import load_dotenv
from tavily import TavilyClient
from llama_parse import LlamaParse
from langchain_core.tools import tool
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from datetime import datetime, timedelta
from amadeus import Client, ResponseError
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage
from llama_index.core import SimpleDirectoryReader
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from IPython.display import Image, display, clear_output

## Configuration

In [1]:
load_dotenv()

print("API Keys configured.")
print(f" - Aval A.I. API Key: {f"{os.getenv('OPENAI_API_KEY')[:3]}****" if os.getenv('OPENAI_API_KEY') else 'NOT SET'}")
print(f" - Tavily API Key: {f"{os.getenv('TAVILY_API_KEY')[:3]}****" if os.getenv('TAVILY_API_KEY') else 'NOT SET'}")
print(f" - Amadeus Client ID: {f"{os.getenv('AMADEUS_CLIENT_ID')[:3]}****" if os.getenv('AMADEUS_CLIENT_ID') else 'NOT SET'}")
print(f" - Amadeus Client Secret: {f"{os.getenv('AMADEUS_CLIENT_SECRET')[:3]}****" if os.getenv('AMADEUS_CLIENT_SECRET') else 'NOT SET'}")
print(f" - LLAMA Parse API Key: {f"{os.getenv('LLAMA_CLOUD_API_KEY')[:3]}****" if os.getenv('LLAMA_CLOUD_API_KEY') else 'NOT SET'}")

API Keys configured.
 - Aval A.I. API Key: aa-****
 - Tavily API Key: tvl****
 - Amadeus Client ID: lJS****
 - Amadeus Client Secret: vM3****
 - LLAMA Parse API Key: llx****


## Dataset Preparation

Download and preprocess the datasets used to build the agent's knowledge and reference tables:
- **IATA Airport Dataset** (Kaggle) — maps city names to IATA airport codes.
- **Currency Country Mapping** (Kaggle) — maps countries to currency codes with fuzzy matching helpers.
- **FAQ set** (`FAQ.js`) — a curated list of travel-agency FAQs.
- **World Travel Book PDF** — parsed via **LlamaParse** into per-page markdown chunks with page titles.

In [2]:
dataset = kagglehub.dataset_download("zinovadr/iata-airport-code") + "/airport-codes_csv.csv"
airports_df = pd.read_csv(dataset)
airports_with_iata = airports_df[airports_df['iata_code'].notna()]

city_to_iata = {}
for _, row in airports_with_iata.iterrows():
    city = str(row['municipality']).strip()
    iata = str(row['iata_code']).strip()
    if city and iata and city != 'nan' and city != 'None' and iata != 'nan':
        city = city.title()

        if city not in city_to_iata:
            city_to_iata[city] = [iata]
        else:
            # select only one airport for Tehran
            if city == "Tehran":
                continue
            if iata not in city_to_iata[city]:
                city_to_iata[city].append(iata)

print(f"Number of airports with IATA codes: {len(airports_with_iata):,}")
print(f"Number of cities with IATA airports: {len(city_to_iata):,}")

def search_city(city_name):
    for city, iata_codes in city_to_iata.items():
        if city_name.lower() in city.lower():
            return iata_codes
    return None

list_cities = ["Dubai", "Tehran", "New York"]
print("\nSample search results:")
for city_name in list_cities:
    iata_codes = search_city(city_name)
    result_airport_code = f"{city_name} → {', '.join(iata_codes)} (Number of airport: {len(iata_codes)})" if iata_codes else f"{city_name} Not found"
    print(result_airport_code)

Using Colab cache for faster access to the 'iata-airport-code' dataset.
Number of airports with IATA codes: 9,199
Number of cities with IATA airports: 7,741

Sample search results:
Dubai → DCG, DXB, NHD (Number of airport: 3)
Tehran → IKA (Number of airport: 1)
New York → TSS, QNY, JPB, JRA, JRB, JFK, LGA (Number of airport: 7)


In [3]:
dataset_path = kagglehub.dataset_download("phanee16/currency-and-country-code-mapping")
country_currency_dataframe = pd.read_csv(dataset_path + "/country_code_to_currency_code.csv")

country_to_currency = {}
for _, row in country_currency_dataframe.iterrows():
    country = str(row['Country']).lower().strip()
    currency_code = str(row['Currency_Code']).strip()

    if country and currency_code and country != 'nan' and country != 'None' and currency_code != 'nan':
        country = country.title()

        if country not in country_to_currency:
            country_to_currency[country] = [currency_code]
        else:
            if currency_code not in country_to_currency[country]:
                country_to_currency[country].append(currency_code)

print(f"Number of Countries: {len(country_currency_dataframe["Country"].unique())}")
print(f"Number of Currencies: {len(country_currency_dataframe["Currency_Code"].unique())}")


# functions to handel different name of countries
def normalize_text(text):
    if not text:
        return ""
    text = re.sub(r'[\W\s]', '', text.strip().lower())
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\b(the|of|and|in)\b', '', text)
    return text.strip()

def create_country_mapping(dataframe):
    mapping = {}

    for _, row in dataframe.iterrows():
        country_name = row.get('Country', '').strip()
        normalized = normalize_text(country_name)
        if normalized not in mapping:
            mapping[normalized] = []
        mapping[normalized].append(row)

        country_code = row.get('country_Code', '').strip().lower()
        if country_code:
            if country_code not in mapping:
                mapping[country_code] = []
            mapping[country_code].append(row)

    return mapping

# Some country names that are not in the dataset to be found
def get_common_country_names():
    return {
        'UNITED ARAB EMIRATES (THE)': ['uae', 'emirates', 'dubai', 'abu dhabi', 'united arab'],
        'UNITED STATES OF AMERICA (THE)': ['usa', 'america', 'us', 'united states', 'the united states'],
        'UNITED KINGDOM OF GREAT BRITAIN AND NORTHERN IRELAND (THE)': ['uk', 'britain', 'england', 'great britain', 'british'],
        'KOREA (THE REPUBLIC OF)': ['south korea', 'korea south', 'korea'],
        'KOREA (THE DEMOCRATIC PEOPLE’S REPUBLIC OF)': ['north korea', 'korea north'],
        'NETHERLANDS (THE)': ['netherlands', 'holland']
    }


def get_country_name(user_input, dataframe, threshold=0.6):
    if not user_input or dataframe.empty:
        return []

    user_input_norm = normalize_text(user_input)
    mapping = create_country_mapping(dataframe)

    if user_input_norm in mapping:
        country_names = set()
        for row in mapping[user_input_norm]:
            country_name = row.get('Country')
            if country_name:
                country_names.add(country_name)
        return list(country_names)

    common_names = get_common_country_names()
    for official_name, variations in common_names.items():
        official_norm = normalize_text(official_name)
        if (user_input_norm == official_norm or
            user_input_norm in [normalize_text(v) for v in variations]):
            if official_norm in mapping:
                country_names = set()
                for row in mapping[official_norm]:
                    country_name = row.get('Country')
                    if country_name:
                        country_names.add(country_name)
                return list(country_names)

    all_country_names = list(mapping.keys())
    matches = difflib.get_close_matches(user_input_norm, all_country_names, n=3, cutoff=threshold)

    if matches:
        country_names = set()
        for row in mapping[matches[0]]:
            country_name = row.get('Country')
            if country_name:
                country_names.add(country_name)
        return list(country_names)

    for index, row in dataframe.iterrows():
        currency_code = row.get('Currency_Code', '').strip().lower()
        currency_name = row.get('Currency Name', '').strip().lower()
        country_code = row.get('country_Code', '').strip().lower()

        if (user_input_norm == currency_code or
            user_input_norm == normalize_text(country_name) or
            user_input_norm == country_code):
            country_name = row.get('Country', '').strip()
            country_norm = normalize_text(country_name)
            if country_norm in mapping:
                country_names = set()
                for row in mapping[country_norm]:
                    country_name = row.get('Country')
                    if country_name:
                        country_names.add(country_name)
                return list(country_names)

    return []

def search_country(country_name):
    country_name_list = get_country_name(country_name, country_currency_dataframe, threshold=0.5)
    if not country_name_list:
        return None
    country_name = country_name_list[0]
    for country, currency_code in country_to_currency.items():
        if country_name.lower() in country.lower():
            return currency_code
    else:
        return None

list_countries = ["Iran", 'UAE', 'United Kingdom', 'United States']

print("\nSample search results:")
for country_item in list_countries:
    currency_code = search_country(country_item)
    result_currency_code = f"{country_item} \u2192 {', '.join(currency_code)} (Number of currency: {len(currency_code)})" if currency_code else f"{country_item} Not found"
    print(result_currency_code)


Using Colab cache for faster access to the 'currency-and-country-code-mapping' dataset.
Number of Countries: 250
Number of Currencies: 167

Sample search results:
Iran → IRR (Number of currency: 1)
UAE → AED (Number of currency: 1)
United Kingdom → EUR (Number of currency: 1)
United States → USD, USN (Number of currency: 2)


In [4]:
with open('FAQ.js', 'r', encoding='utf-8') as f:
    faq_data = json.load(f)

print(f"Number of FAQs: {len(faq_data)}")
print("\nSample FAQ:")
for i, item in enumerate(faq_data[:3]):
    print(f"{i+1}. Q: {item['question']}")
    print(f"   A: {item['answer']}\n")

Number of FAQs: 30

Sample FAQ:
1. Q: What documents do I need for international travel?
   A: For international travel, you typically need: 1) A valid passport (valid for at least 6 months), 2) Visa (if required for your destination), 3) Travel insurance, 4) Vaccination certificates (if required), 5) Return ticket.

2. Q: How early should I arrive at the airport?
   A: For international flights, arrive 3 hours before departure. For domestic flights, arrive 2 hours before departure. This allows time for check-in, security, and any unexpected delays.

3. Q: What items are not allowed in carry-on luggage?
   A: Prohibited items in carry-on include: liquids over 100ml, sharp objects (knives, scissors), flammable materials, weapons, and lithium batteries over 100Wh. Check your airline's specific restrictions.



In [6]:
if not os.getenv("LLAMA_CLOUD_API_KEY"):
    raise ValueError("LLAMA_CLOUD_API_KEY does not saved!")

def extract_title_from_text(text, page_num):
    lines = text.split('\n')
    for line in lines[:5]:
        line = line.strip()
        if line and len(line) < 200:
            clean_line = re.sub(r'^\d+[\.\s]*', '', line)
            if clean_line and len(clean_line.split()) < 10:
                return clean_line
    return None

travel_book_filename = "./World Travel Book.pdf"
if not os.path.exists(travel_book_filename):
    raise ValueError(f"File {travel_book_filename} doesn't exist.")

parse_time = datetime.now()
print(f"Parsing '{travel_book_filename}'...")
parser = LlamaParse(
    result_type="markdown",
    verbose=True,
    user_prompt="Extract the title of each page if it exists and include it in the metadata with key 'page_title'. Then, extract the rest of the content as text.",
    language="en"
)
documents = parser.load_data(travel_book_filename)

if not documents:
    raise ValueError("LlamaParse doesn't work properly.")

book_data = []
for number, document in enumerate(documents, 1):
    page_title = document.metadata.get('page_title')
    if not page_title:
        page_title = extract_title_from_text(document.text, number)
    if not page_title and last_title:
        page_title = f"{last_title} (continued)"
    elif not page_title:
        page_title = f"Page {number}"
    if page_title and not page_title.startswith("Page ") and "(continued)" not in page_title:
        last_title = page_title
    book_data.append({
        "page_number": number,
        "title": page_title,
        "text": document.text,
        "creation_date": parse_time,
        "source": travel_book_filename
    })

print(f"Number of Page Parsed: {len(book_data)}")
print("\nSample of the book:")
for item in book_data[:3]:
    print(f"Page Title: {item['title']}")
    print(f"Page number: {item['page_number']}")
    print(f"Page text: {item.get('text')[:50]}")
    print(f"Creation date: {item['creation_date']}")


Parsing './World Travel Book.pdf'...
Started parsing the file under job_id b5d481e9-73fc-4a64-bb05-7d80d5ebe0f7
Number of Page Parsed: 433

Sample of the book:
Page Title: # ANTHONY BOURDAIN
Page number: 1
Page text: # ANTHONY BOURDAIN AND LAURIE WOOLEVER QUANT TRAVE
Creation date: 2026-02-09 19:24:25.865688
Page Title: # WORLD TRAVEL an irreverent guide
Page number: 2
Page text: 
# WORLD TRAVEL an irreverent guide

# WORLD TRAVE
Creation date: 2026-02-09 19:24:25.865688
Page Title: An Imprint of HarperCollinsPublishers
Page number: 3
Page text: 
An Imprint of HarperCollinsPublishers

Creation date: 2026-02-09 19:24:25.865688


## Vector Database Construction (LanceDB)

Build a **LanceDB** vector store with two tables:
- `faq` — question/answer pairs embedded via `BAAI/bge-small-en-v1.5`.
- `book` — page-level chunks of the parsed travel book with page number, title, and source metadata.

Includes sample semantic searches over both tables.

In [7]:
faq_table_name = "faq"
book_table_name = "book"
model_name = "BAAI/bge-small-en-v1.5"
registry = get_registry()
func = registry.get("huggingface").create(name=model_name)
db = lancedb.connect("./lancedb_travel_bot")

class FAQSchema(LanceModel):
    question: str = func.SourceField()
    answer: str
    vector: Vector(func.ndims()) = func.VectorField()

class BookSchema(LanceModel):
    page_number: int
    title: str
    text: str = func.SourceField()
    creation_date: str
    source: str
    vector: Vector(func.ndims()) = func.VectorField()

if not faq_data:
    raise ValueError("FAQ.js file does not load properly.")

faq_table = db.create_table(faq_table_name, schema=FAQSchema, mode="overwrite")
faq_table.add(faq_data)
print(f"FAQ Table created and {len(faq_data)} records added in {faq_table_name} table.")

if not book_data:
    raise ValueError(f"The {travel_book_filename} doesn't parse correctly.")

book_table = db.create_table(book_table_name, schema=BookSchema, mode="overwrite")
book_table.add(book_data)
print(f"Book Table created and {len(book_data)} records added in {book_table_name} table.")

def db_search_in(table_name, query, limit=1):
    table = db.open_table(table_name)
    results = table.search(query).limit(limit).to_pandas()
    if not results.empty:
        return results

    return None

queries = [
    {"query": "Change my flight after booking", "table": faq_table_name},
    {"query": "Documents for international travel", "table": faq_table_name},
    {"query": "How can I cancel my flight?", "table": faq_table_name},
    {"query": "Which Satate in United States of America is good for traveling", "table": book_table_name},
    {"query": "Paris", "table": book_table_name},
    {"query": "Information about historical places in Paris", "table": book_table_name},
]

print("\nSample test search in the tables:")
for item in queries:
    search_result = db_search_in(item.get('table'), item.get('query'), limit=1)
    print(f"\n - Query: {item.get('query')}")

    if not search_result.empty:
        if item.get('table') == 'faq':
            print(f" - Question: {search_result.iloc[0]['question']}")
        elif item.get('table') == 'book':
            print(f" - Book text: {repr(search_result.iloc[0]['text'][:200])}")
    else:
        print(f" - No {'FAQ' if item.get('table') == 'faq' else 'Book context'} found.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


FAQ Table created and 30 records added in faq table.


Book Table created and 433 records added in book table.

Sample test search in the tables:



 - Query: Change my flight after booking
 - Question: Can I change my flight after booking?



 - Query: Documents for international travel
 - Question: What documents do I need for international travel?



 - Query: How can I cancel my flight?
 - Question: What should I do if my flight is delayed or cancelled?



 - Query: Which Satate in United States of America is good for traveling
 - Book text: '\n\n\n# America’s most important chefs\n\nA guy who is redefining not just what southern cooking is, was, and can be, but American cooking as a whole.\n\n# ARRIVAL AND GETTING AROUND\n\nCharleston Internationa'



 - Query: Paris
 - Book text: '\n\n\n# Paris\n\nTony loved Paris. He first visited as a child, as he detailed in Kitchen Confidential, and he made the first and 100th episodes of No Reservations there, along with an episode of The Layov'



 - Query: Information about historical places in Paris
 - Book text: '\n\n\n“Now, here’s the thing about hotels in Paris: you can go the Henry Miller route and wallow romantically in squalor, or live it up big time. For a luxurious, high-end option, there’s the elegantly d'


## Tool Definitions

Register the LangChain **tools** used by the agent — all wrapped with `@tool` so they can be bound to the LLM:
- `flight_search` — Amadeus flight offers search.
- `hotel_search` — Amadeus hotel offers search with a Tavily fallback.
- `restaurant_search` — Tavily-based restaurant discovery.
- `get_weather_info` — Tavily-based weather lookup.
- `get_currency_info` — Tavily-based exchange rate lookup.
- `faq_search` — semantic FAQ retrieval from LanceDB.
- `trip_planning` — hybrid (LanceDB + Tavily) day-by-day itinerary planner.

In [8]:
if not os.getenv("AMADEUS_CLIENT_ID"):
    raise ValueError("AMADEUS_CLIENT_ID does not saved!")

if not os.getenv("AMADEUS_CLIENT_SECRET"):
    raise ValueError("AMADEUS_CLIENT_SECRET does not saved!")

if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY does not saved!")

amadeus = Client(client_id=os.environ.get("AMADEUS_CLIENT_ID"), client_secret=os.environ.get("AMADEUS_CLIENT_SECRET"))

tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

### Flight Search Tool

Searches one-way flight offers between two cities on a given date using Amadeus.

In [9]:
@tool
def flight_search(origin_city, destination_city, departure_date):
    """
    Searches for flights between two cities on a specific date using Amadeus API.

    Args:
        origin_city (str): The name of the city departing from.
        destination_city (str): The name of the destination city.
        departure_date (str): The date of departure in 'YYYY-MM-DD' format.
    """
    try:
        get_city_code = lambda city_name: city_to_iata.get(city_name.strip().title())
        origin_codes = get_city_code(origin_city)
        destination_codes = get_city_code(destination_city)

        if not origin_codes:
            return f"Could not find IATA code for {origin_city}."
        if not destination_codes:
            return f"Could not find IATA code for {destination_city}."

        format_date = lambda date_sting: datetime.strptime(re.sub(r'[./ ]', '-', date_sting), '%Y-%m-%d').date().isoformat()

        try:
            format_departure_date = format_date(departure_date)
        except ValueError:
            return "Invalid date format. Please use YYYY-MM-DD."

        flights = []
        for origin in origin_codes:
            for destination in destination_codes:
                try:
                    response = amadeus.shopping.flight_offers_search.get(
                        originLocationCode=origin,
                        destinationLocationCode=destination,
                        departureDate=format_departure_date,
                        adults=1,
                        max=3
                    )

                    if response and hasattr(response, 'data') and response.data:
                        flights.extend(response.data)

                    time.sleep(0.5)
                except (Exception, ResponseError) as e:
                    continue

        if not flights:
            raise Exception(f"No flights found from {origin_city} to {destination_city} on {format_departure_date}.")

        results = []
        for idx, flight in enumerate(flights, 1):
            price = flight['price']['total']
            currency = flight['price']['currency']
            itineraries = flight['itineraries'][0]['segments']
            stops = len(itineraries) - 1

            airline_code = itineraries[0]['carrierCode']
            departure_time = itineraries[0]['departure']['at']
            duration = flight['itineraries'][0]['duration']
            flight_number = itineraries[0]['flightNumber']

            hours, minutes = duration.split("H")
            hours = f"{hours[2:]} hours" if hours[2:].isdigit() and int(hours[2:]) > 1 else f"{hours[2:]} hour" if hours[2:].isdigit() and int(hours[2:]) == 1 else ""
            minutes = f" and {minutes[:-1]} minutes" if minutes[:-1].isdigit() and int(minutes[:-1]) > 1 else f" and {minutes[:-1]} mintue" if minutes[:-1].isdigit() and int(minutes[:-1]) == 1 else ""
            duration = hours + minutes

            result = f"""Filght Number {idx}\
             \n - Airline: {airline_code}\format_date = lambda date_sting: datetime.strptime(re.sub(r'[./ ]', '-', date_sting), '%Y-%m-%d').date().isoformat()
             \n - Airport: from {itineraries[0]['departure']['iataCode']} to {itineraries[-1]['arrival']['iataCode']}\
             \n - Price: {price} {currency}\
             \n - Departs: {departure_time.split('T')[0]} date at {departure_time.split('T')[1]}\
             \n - Duration: {duration}\
             \n - Stops: {stops}"""

            results.append(result)

        return "\n\n".join(results)

    except (ResponseError, Exception) as e:
        return f"Error: {e}"

flight_search_queries = [
    {"origin_city": "Tehran", "destination_city": "Dubai", "departure_date": "2026-03-10"},
    {"origin_city": "Paris", "destination_city": "Barcelona", "departure_date": "2026-03-12"}
]

print("Flights from origin to destination on date:")
for idx, item in enumerate(flight_search_queries, 1):
    print(f"\033[1mSerch Flight {idx} From {item.get('origin_city')} to {item.get('destination_city')} on {item.get('departure_date')}\033[0m")
    result = flight_search.invoke(item)
    print(result, "\n")


Flights from origin to destination on date:
Serch Flight 1 From Tehran to Dubai on 2026-03-10
Filght Number 1             
 - Airline: X1             
 - Airport: from IKA to DXB             
 - Price: 178.80 EUR             
 - Departs: 2026-03-10 date at 18:05:00             
 - Duration: 15 hours and 30 minutes             
 - Stops: 1

Filght Number 2             
 - Airline: X1             
 - Airport: from IKA to DXB             
 - Price: 178.80 EUR             
 - Departs: 2026-03-10 date at 18:05:00             
 - Duration: 24 hours and 30 minutes             
 - Stops: 1

Filght Number 3             
 - Airline: QR             
 - Airport: from IKA to DXB             
 - Price: 185.54 EUR             
 - Departs: 2026-03-10 date at 05:00:00             
 - Duration: 6 hours and 10 minutes             
 - Stops: 1 

Serch Flight 2 From Paris to Barcelona on 2026-03-12
Filght Number 1             
 - Airline: IB             
 - Airport: from CDG to BCN             
 - Price: 1

### Hotel Search Tool

Searches for hotels in a city for a given check-in/check-out range via Amadeus, with a Tavily fallback on failure.

In [10]:
@tool
def hotel_search(destination_city, check_in_date, check_out_date, limit=5):
    """
    Searches for hotels in a specific city for a given date range.

    Args:
        destination_city (str): The name of the city.
        check_in_date (str): Check-in date in 'YYYY-MM-DD' format.
        check_out_date (str): Check-out date in 'YYYY-MM-DD' format.
    """
    try:
        get_city_code = lambda city_name: city_to_iata.get(city_name.strip().title())
        destination_code = get_city_code(destination_city)

        if not destination_code:
            return f"Could not find IATA code for {destination_city}."

        format_date = lambda date_sting: datetime.strptime(re.sub(r'[./ ]', '-', date_sting), '%Y-%m-%d').date().isoformat()

        try:
            format_check_in_date = format_date(check_in_date)
            format_check_out_date = format_date(check_out_date)
        except:
            return "Invalid date format. Please use YYYY-MM-DD."

        hotels = []
        for destination in destination_code:
            try:
                hotels_by_city = amadeus.reference_data.locations.hotels.by_city.get(cityCode=destination)
                hotelIds = [hotel.get('hotelId') for hotel in hotels_by_city.data[:limit]]
                hotel_offers = amadeus.shopping.hotel_offers_search.get(
                    hotelIds=hotelIds,
                    checkInDate=format_check_in_date,
                    checkOutDate=format_check_out_date,
                    adults='1'
                )

                if hotel_offers and hasattr(hotel_offers, 'data') and hotel_offers.data:
                    hotels.extend(hotel_offers.data)

                time.sleep(0.5)
            except (Exception, ResponseError) as e:
                continue

        if not hotels:
            raise Exception(f"No hotel found with amadeus in {destination_city}")

        results = []
        for idx, offer in enumerate(hotels, 1):
            hotel_info = offer['hotel']
            main_offer = offer['offers'][0]
            price = main_offer['price']['total']
            currency = main_offer['price']['currency']
            rating = hotel_info['rating']
            description = repr(main_offer['room']['description']['text'])

            result = f"""Hotel Number {idx}\
            \n - Name: {hotel_info['name']}\
            \n - Price: {price} {currency}\
            \n - Description: {description}\
            \n - Address: Latitude: {hotel_info['latitude']}, Longitude: {hotel_info['longitude']}"""
            results.append(result)

        return "\n\n".join(results)

    except (ResponseError, Exception) as e:
        print(f"Error: {e}")
        query = f"best hotels in {destination_city} between {check_in_date} and {check_out_date}"
        responses = tavily.search(query=query, search_depth="basic", max_results=limit)
        results = []
        for idx, response in enumerate(responses['results'], 1):
            result = f"""Hotel Number {idx}\
                    \n - Title: {response['title']}\
                    \n - Content: {response['content']}\
                    \n - URL: {response['url']}"""
            results.append(result)

        return "\n\n".join(results)

hotel_search_queries = [
    {"destination_city": "Dubai", "check_in_date": "2026-03-10", "check_out_date": "2026-03-15"},
    {"destination_city": "Paris", "check_in_date": "2026-03-10", "check_out_date": "2026-03-15"}
]

print("Hotels in the city duration specfic time:")
for idx, item in enumerate(hotel_search_queries, 1):
    print(f"\033[1mSerch Hotel {idx} in {item.get('destination_city')} between {item.get('check_in_date')} and {item.get('check_out_date')}\033[0m")
    result = hotel_search.invoke(item)
    print(result, "\n")


Hotels in the city duration specfic time:
Serch Hotel 1 in Dubai between 2026-03-10 and 2026-03-15
Hotel Number 1            
 - Name: InterContinental Marina            
 - Price: 5269.51 AED            
 - Description: 'BEST FLEXIBLE RATE\nClassic Room When you arrive at the hotel we\nwill do our best to meet your room type'            
 - Address: Latitude: 51.21983, Longitude: 6.7784 

Serch Hotel 2 in Paris between 2026-03-10 and 2026-03-15
Hotel Number 1            
 - Name: Best Western Plus l’Oree Paris Sud            
 - Price: 828.50 EUR            
 - Description: 'FLEXIBLE RATE*BEST LEAST RESTRICTIVE RATE\n1KING 2 SINGLE BEDS,NSMK,FAMROO,2EXTBD,BATHTB\nSTANDARD'            
 - Address: Latitude: 48.67518, Longitude: 2.2797 



### Restaurant Search Tool

Searches for the best local restaurants and cuisine in a destination using Tavily.

In [12]:
@tool
def restaurant_search(destination, limit=5):
    """
    Searches for the best restaurants and local food in a destination.

    Args:
        destination (str): The city or location name.
    """
    query = f"best restaurants and local cuisine in {destination}"
    try:
        responses = tavily.search(query=query, search_depth="basic", max_results=limit)
        if not responses['results']:
            raise Exception(f"Could not find any restaurant for '{destination}' at the moment.")

        results = []
        for idx, response in enumerate(responses['results'], 1):
            result = f"""Restaurant Number {idx}\
            \n - Title: {response['title']}\
            \n - Content: {response['content']}\
            \n - URL: {response['url']}"""
            results.append(result)

        return "\n\n".join(results)

    except Exception as e:
        return f"Error: {e}"

restaurant_search_queries = [
    {"destination": "Paris"},
    {"destination": "London"}
]

print("Restaurants in the city:")
for idx, item in enumerate(restaurant_search_queries, 1):
    print(f"\033[1mSerch Restaurant {idx} in {item.get('destination')}\033[0m")
    print(f"City {idx}: {item.get('destination')}")
    result = restaurant_search.invoke(item)
    print(result, "\n")

Restaurants in the city:
City 1: Paris

Restaurant Number 1            
 - Title: What are the best local authentic restaurants in Paris?            
 - Content: 1. Bouillon Pigalle District: Pigalle ✨ A local favorite for classic French dishes like escargots and crème brûlée. Great vibes and            
 - URL: https://www.facebook.com/groups/RickStevesEurope/posts/1672330890149194/

Restaurant Number 2            
 - Title: 25 delicious food tips in Paris from top bloggers and locals            
 - Content: Easy and affordable eating · 1. Le Puits de Légumes · 2. Le Relais Gascon in Montmartre · 3. Opium · 4. Rue du Faubourg St Denis · 5. Parisian Crepes.            
 - URL: https://www.heatheronhertravels.com/blogger-food-tips-paris/

Restaurant Number 3            
 - Title: Where to Eat in Paris: My Favourite Restaurants            
 - Content: Where to Eat in Paris: My Favourite Restaurants · 1. Hotel Costes · 2. La Belle Epoque · 3. LouLou · 4. Bar des Pres · 5. Lapérouse · 5. S

### Weather Info Tool

Fetches weather forecast details for a destination and date using Tavily (parses JSON returned in the result content).

In [13]:
@tool
def get_weather_info(destination, predict_date, limit=1):
    """
    Fetches weather forecast information for a specified destination and date.

    Args:
        destination (str): The city or location name.
        predict_date (str): The date for which to fetch weather forecast.
    """
    query = f"weather forecast in {destination} on {predict_date} and three next days"
    try:
        responses = tavily.search(query=query, search_depth="basic", max_results=limit)

        weather_details = responses['results'][0]['content'].replace("'", '"')
        weather_details = json.loads(weather_details)

        if not weather_details:
            raise Exception(f"Could not find weather information for {destination} on {predict_date}.")

        result = f""" - Country: {weather_details['location']['country']}\
            \n - Region: {weather_details['location']['region']}\
            \n - Last Update: {weather_details['current']['last_updated']}\
            \n - Temperature: {weather_details['current']['temp_c']}°C\
            \n - Condition: {weather_details['current']['condition']['text']}\
            \n - Humidity: {weather_details['current']['humidity']}%\
            \n - Could: {weather_details['current']['cloud']}%\
            \n - Wind: {weather_details['current']['wind_kph']}km/h"""

        return result

    except Exception as e:
        return f"Error: {e}"

get_weather_info_queries = [
    {"destination": "Dubai", "predict_date": "January 16 2026"},
    {"destination": "Paris", "predict_date": "January 16 2026"}
]

print("Weather Information in the city:")
for idx, item in enumerate(get_weather_info_queries, 1):
    print(f"\033[1mWeather {idx} in {item.get('destination')} on {item.get('predict_date')}\033[0m")
    result = get_weather_info.invoke(item)
    print(result, "\n")


Weather Information in the city:
Weather 1 in Dubai on January 16 2026
 - Country: United Arab Emirates            
 - Region: Dubai            
 - Last Update: 2026-02-09 23:15            
 - Temperature: 23.1°C            
 - Condition: Clear            
 - Humidity: 78%            
 - Could: 0%            
 - Wind: 15.8km/h 

Weather 2 in Paris on January 16 2026
 - Country: France            
 - Region: Ile-de-France            
 - Last Update: 2026-02-09 20:15            
 - Temperature: 10.2°C            
 - Condition: Patchy rain nearby            
 - Humidity: 71%            
 - Could: 0%            
 - Wind: 17.3km/h 



### Currency Info Tool

Resolves currency codes from country names and fetches the exchange rate via Tavily.

In [14]:
@tool
def get_currency_info(origin_country, destination_country):
    """
    Gets the currency exchange rate between two countries.

    Args:
        origin_country (str): Name of the origin country.
        destination_country (str): Name of the destination country.
    """
    try:
        get_currency_code = lambda country_name: country_to_currency.get(country_name.strip().title())
        origin_code = get_currency_code(origin_country)
        destination_code = get_currency_code(destination_country)

        if not origin_code:
            raise Exception(f"Could not find currency for {origin_country}.")

        if not destination_code:
            raise Exception(f"Could not find currency for {destination_country}.")

        origin_code = origin_code[0]
        destination_code = destination_code[0]

        query = f"exchange rate {origin_code} to {destination_code}"
        responses = tavily.search(query=query, max_results=1)

        results = []
        for response in responses['results']:
            result = f"""\
            \n - Title: {response['title']}\
            \n - Content: {response['content']}\
            \n - URL: {response['url']}"""
            results.append(result)

        return "\n\n".join(results)

    except Exception as e:
        return f"Error: {e}"

get_currency_info_queries = [
    {"origin_country": "UNITED STATES OF AMERICA (THE)", "destination_country": "UNITED ARAB EMIRATES (THE)"},
    {"origin_country": "France", "destination_country": "Japan"}
]

print("Get Currency Exchange Rate:")
for idx, item in enumerate(get_currency_info_queries, 1):
    print(f"\033[1mSample {idx}: form {item.get('origin_country')} to {item.get('destination_country')}\033[0m")
    result = get_currency_info.invoke(item)
    print(result, "\n")


Get Currency Exchange Rate:
Sample 1: form UNITED STATES OF AMERICA (THE) to UNITED ARAB EMIRATES (THE)
            
 - Title: 1 USD to AED - US Dollars to Emirati Dirhams Exchange Rate - Xe            
 - Content: As of 04:01 UTC, the mid-market USD to AED rate is $1 = د.إ3.6725. The mid-market rate is the midpoint between buy and sell prices in global currency markets.            
 - URL: https://www.xe.com/en-us/currencyconverter/convert/?Amount=1&From=USD&To=AED 

Sample 2: form France to Japan
            
 - Title: 1 EUR to JPY - Euros to Japanese Yen Exchange Rate - Xe            
 - Content: As of 09:17 UTC, the mid-market EUR to JPY rate is €1 = ¥185.63. The mid-market rate is the midpoint between buy and sell prices in global currency markets. To            
 - URL: https://www.xe.com/en-us/currencyconverter/convert/?Amount=1&From=EUR&To=JPY 



### FAQ Search Tool

Semantic search over the FAQ vector table in LanceDB.

In [16]:
@tool
def faq_search(query):
    """
    Searches the travel agency's FAQ database for answers to common questions.

    Args:
        query (str): The user's question or query.
    """
    try:
        responses = faq_table.search(query).limit(2).to_pandas()

        if responses.empty:
            raise Exception("No relevant FAQ found.")

        results = []
        for _, row in responses.iterrows():
            result = f""" - Question: {row['question']}\
            \n - Answer: {row['answer']}"""

            results.append(result)

        return '\n\n'.join(results)

    except Exception as e:
        return f"Error: {e}"

faq_search_queries = [
    {"query": "What documents do I need for international travel?"},
    {"query": "Can I take food on the plane?"},
    {"query": "How early should I arrive at airport?"}
]

for idx, item in enumerate(faq_search_queries, 1):
    print(f"\033[1mSample {idx}: {item.get('query')}\033[0m")
    result = faq_search.invoke(item)
    print(result, "\n")


Sample 1: What documents do I need for international travel?
 - Question: What documents do I need for international travel?            
 - Answer: For international travel, you typically need: 1) A valid passport (valid for at least 6 months), 2) Visa (if required for your destination), 3) Travel insurance, 4) Vaccination certificates (if required), 5) Return ticket.

 - Question: Are there any travel restrictions I should know?            
 - Answer: Check current COVID-19 restrictions, visa requirements, political situation, and health advisories for your destination. Visit your government's travel advisory website before booking. 

Sample 2: Can I take food on the plane?
 - Question: Can I take food on the plane?            
 - Answer: Solid foods are generally allowed in carry-on. Liquids, gels, and spreads must follow the 100ml rule. International flights may restrict certain foods (fruits, meat, dairy) at the destination.

 - Question: Can I bring my pet on the plane?           

### Trip Planning Tool

Combines semantic search on the book table (with a Tavily fallback) to gather context for a day-by-day itinerary generation prompt.

In [18]:
@tool
def trip_planning(destination, arrival_date, departure_date, interests="general"):
    """
    Creates a travel planning based on the destination and travel dates.

    Args:
        destination (str): City or Country to visit.
        arrival_date (str): YYYY-MM-DD
        departure_date (str): YYYY-MM-DD
        interests (str): Optional user interests.
    """
    itinerary = []

    try:
        start = datetime.strptime(arrival_date, "%Y-%m-%d")
        end = datetime.strptime(departure_date, "%Y-%m-%d")
        duration = (end - start).days + 1
        if duration <= 0:
            raise Exception("The departure date cannot be before the arrival date.")
    except Exception as e:
        return f"Error: {e}"

    queries = [
        f"main attractions and sightseeing in {destination}",
        f"best local food and restaurants in {destination}",
        f"activities and things to do in {destination} for {interests}"
    ]

    book_context = []
    try:
        for query in queries:
            results = book_table.search(query).limit(3).to_pandas()

            if results.empty:
                print(f"No relevant results found in LanceDB for '{query}'. Attempting web search as fallback.")
                web_query = f"{destination} {query}"
                web_results = tavily.search(query=web_query, max_results=1)
                if web_results and web_results.get('results'):
                    book_context.append(web_results['results'][0]['content'])
                else:
                    print(f"No web results found for '{web_query}'.")
            else:
                for _, row in results.iterrows():
                    if 'text' in row and pd.notna(row['text']):
                        book_context.append(row['text'])

        if len(book_context) < len(queries):
            print("Context from travel book is sparse. Supplementing with general web search.")
            web_query = f"{duration} day itinerary for {destination} focusing on {interests}"
            web_results = tavily.search(query=web_query, max_results=3)
            if web_results and web_results['results']:
                for res in web_results['results']:
                    if 'content' in res and pd.notna(res['content']):
                        book_context.append(res['content'])

    except Exception as e:
        return f"Error during context gathering for trip planning: {e}"

    resluts = f"""Destination: {destination}\
        \n - From {arrival_date} to {departure_date} ({duration} days)\
        \n - Focusing on {interests}\
        \n - Context for Planning: {repr('\n'.join(book_context)[:1000])}\
        \n - Instruction: Please create a detailed Day-by-Day itinerary based on the context above. Include morning, afternoon, and evening activities for each of the {duration} days."""

    return resluts

trip_planning_queries = [
    {"destination": "Paris", "arrival_date": "2026-05-10", "departure_date": "2026-05-14", "interests": "history and culture"},
    {"destination": "London", "arrival_date": "2026-03-10", "departure_date": "2026-03-15", "interests": "fun and food"}
]

print("Trip Planning Queries:")
for idx, item in enumerate(trip_planning_queries, 1):
    print(f"\033[1mSample {idx}: {item.get('destination')} from {item.get('arrival_date')} to {item.get('departure_date')} with interests {item.get('interests')}\033[0m")
    result = trip_planning.invoke(item)
    print(result, "\n")


Trip Planning Queries:
Sample 1: Paris from 2026-05-10 to 2026-05-14 with interests history and culture
Destination: Paris        
 - From 2026-05-10 to 2026-05-14 (5 days)        
 - Focusing on history and culture        
 - Context for Planning: '\n\n\nThere’s also a shuttle bus to central Paris (about 18 euros/US$20; see www.lebusdirect.com/en).\n\nIf you fly into Orly (ORY), a slightly smaller international hub that’s about eight miles south of the city, take a taxi (35–40 euros/US$39–$45, with 10 percent tip for good service) to your hotel, or the OrlyVal train to Antony station, to the RER B train to central Paris, or take the aforementioned shuttle bus.\n\nIf you’re coming from elsewhere in France or greater Europe by rail, you’ll arrive at one of Paris’s seven major train stations (the Gares du Nord, de l’Est, d’Austerlitz, de Bercy, de Lyon, Montparnasse, and Saint-Lazare), each of which has a taxi stand and is connected to the vast, efficient Paris Métro (subway) system.\n\n

## LangGraph Agent

Bind all tools to a `gpt-4o-mini` LLM hosted via AvalAI, define a `reasoner` node that invokes the LLM with a system prompt, and compile a state graph with a `tools` node connected using `tools_condition`.

In [19]:
llm = ChatOpenAI(base_url="https://api.avalai.ir/v1", model="gpt-4o-mini", temperature=0.5)

all_tools = [flight_search, hotel_search, restaurant_search, get_weather_info, get_currency_info, faq_search, trip_planning]
llm_with_tools = llm.bind_tools(all_tools)
current_date = datetime.now().strftime("%Y-%m-%d")
system_prompt = f"""You are a professional Travel Assistant agent. Your goal is to help users plan their trips efficiently.
Current Date: {current_date} (Today).
**Guidelines for Tool Usage:**
1. **Flight & Hotel Search:** Always convert city names to English. If the user provides a date like 'next Friday', calculate the exact YYYY-MM-DD based on today's date.
2. **Currency Info:** Always use country names to find exchange rates.
3. **FAQ Search:** Use this for questions about agency policies, baggage, or cancellations.
4. **Trip Planning:** Use this for creating detailed day-by-day itineraries. It uses a vector database of a Travel Book.
5. **Weather:** Always provide clothing tips based on the weather results.
**Response Style:**
- Be polite, helpful, and concise.
- Use emojis to make the conversation engaging.
- If an API fails, explain it gracefully and use alternative information if available."""

print(f"{len(all_tools)} tools bounded successfully on initialized LLM.")

7 tools bounded successfully on initialized LLM.


In [20]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def reasoner_node(state: State):
    return {"messages": [llm_with_tools.invoke([SystemMessage(content=system_prompt)] + state['messages'])]}

print("Reasoner node defined successfully.")

Reasoner node defined successfully.


In [21]:
workflow = StateGraph(State)
workflow.add_node("reasoner", reasoner_node)
workflow.add_node("tools", ToolNode(all_tools))
workflow.add_edge(START, "reasoner")
workflow.add_conditional_edges("reasoner", tools_condition)
workflow.add_edge("tools", "reasoner")
app = workflow.compile()

try:
    display(Image(app.get_graph().draw_mermaid_png()))

except Exception as e:
    print(f"Error: {e}")

### Agent quick test

Helper `travel_bot_answer` that streams the LangGraph app and prints which node/tool is being used; runs two sample queries.

In [ ]:
def travel_bot_answer(query):
    inputs = {"messages": [("user", query)]}
    for output in app.stream(inputs):
        for key, value in output.items():
            print(f" - Using Node: {key.title()}")
            if key == "tools":
                for tool_msg in value["messages"]:
                    print(f"   - Using Tool: {tool_msg.name}")

    final_msg = value["messages"][-1].content

    return final_msg

some_questions = [
    "Paris weather",
    "Best restaurant in Japan tokoyo"
]

for question in some_questions:
    print(f"\033[1mQuestion: {question} \033[0m")
    response = travel_bot_answer(question)
    print(response, "\n\n")


Question: Paris weather 
 - Using Node: Reasoner
 - Using Node: Tools
   - Using Tool: get_weather_info
 - Using Node: Reasoner
The weather in Paris today is overcast with a temperature of around 10.4°C. 🌥️ The humidity is at 71%, and there's a light wind blowing at 15.8 km/h.

### Clothing Tips:
- Dress in layers to stay warm, as it can feel cooler with the wind.
- A light jacket or sweater would be ideal, along with comfortable shoes for walking. Don't forget an umbrella just in case! ☔️ 


Question: Best restaurant in Japan tokoyo 
 - Using Node: Reasoner
 - Using Node: Tools
   - Using Tool: restaurant_search
 - Using Node: Reasoner
Here are some of the best restaurants you can enjoy in Tokyo, Japan:

1. **Loiseau de France** - A fine dining experience with exquisite French cuisine.
   - [Michelin Guide Tokyo Restaurants](https://guide.michelin.com/us/en/tokyo-region/tokyo/restaurants)

2. **B-kyū Gurume Restaurants** - These are local favorites known for their delicious and afford

## CLI Interface

A simple REPL-style CLI that maintains conversation history per session (`thread_id`), supports `clear`/`cls` to reset the conversation, and exits on `exit`/`quit`/`q`.

In [ ]:
def chat_cli():
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    history = []

    welcome_massage = f"""
        \033[1mWelcome to TravelBot CLI!\033[0m
        Type 'exit' or 'quit' or 'q' to quit and to reset the conversation type 'clear' or 'cls'.
        {'-' * 90}"""
    print(welcome_massage)


    while True:
        user_input = input("\nType Your Command or Prompt:").strip()

        if user_input.lower() in ["exit", "quit", "q"]:
            print("Thanking you for using our TravelBot CLI and Goodbye.")
            break

        if user_input.lower() in ["clear", "cls"]:
            history = []
            config = {"configurable": {"thread_id": str(uuid.uuid4())}}
            print("Conversation history cleared.")
            time.sleep(0.8)
            clear_output(wait=True)
            print(welcome_massage)
            continue

        if not user_input:
            continue
        history.append(("user", user_input))
        print("TravelBot is thinking...")
        current_state = {"messages": history}

        for event in app.stream(current_state, config):
            for node_name, value in event.items():
                if node_name == "tools":
                    for tool_msg in value["messages"]:
                        print(f" - Using Tool: {tool_msg.name}")

                if "messages" in value:
                    history.extend(value["messages"])

        chatbot_response = history[-1].content
        print(f"\033[1mTravel-Bot:\033[0m\n{chatbot_response}\n" + "-"*30)

chat_cli()


        Welcome to TravelBot CLI!
        Type 'exit' or 'quit' or 'q' to quit and to reset the conversation type 'clear' or 'cls'.
        ------------------------------------------------------------------------------------------

Type Your Command or Prompt:Suggest flight from Paris To Barcelona
TravelBot is thinking...
 - Using Tool: flight_search
Travel-Bot:
Here are some flight options from Paris to Barcelona on February 10, 2026:

✈️ **Flight Options:**

1. **Airline:** Iberia (IB)  
   **Departure:** 15:25 from Charles de Gaulle Airport (CDG)  
   **Duration:** 1 hour 45 minutes  
   **Price:** €105.54  
   **Stops:** 0

2. **Airline:** Iberia (IB)  
   **Departure:** 09:50 from Charles de Gaulle Airport (CDG)  
   **Duration:** 1 hour 50 minutes  
   **Price:** €105.54  
   **Stops:** 0

3. **Airline:** Iberia (IB)  
   **Departure:** 22:05 from Charles de Gaulle Airport (CDG)  
   **Duration:** 1 hour 50 minutes  
   **Price:** €105.54  
   **Stops:** 0

4. **Airline:** Vueli

## Scenario Testing

Run the agent against 16 different test prompts covering each tool individually plus combined multi-tool scenarios.

In [28]:
scenarios = [
    # 1. Flight Search
    "Find flights from New York to London on March 15th, 2026",
    "I need to fly from Tokyo to Sydney next Friday",

    # 2. Hotel Search
    "Search for hotels in Dubai from March 10th to 17th 2026",
    "Find hotels in the best place of Paris for next weekend",

    # 3. Restaurant Search
    "What are the best local restaurants in Bangkok?",
    "Find top-rated Italian restaurants in Rome",

    # 4. Weather Info
    "What's the weather like in Tokyo this weekend?",
    "Check weather forecast for Istanbul on February 20th 2026",

    # 5. Currency Info
    "What's the exchange rate from US Dollars to Japanese Yen?",
    "Convert China currency to Brazilian Reals",

    # 6. FAQ Search
    "What documents do I need for international travel?",
    "Can I bring my medication on an international flight?",

    # 7. Trip Planning
    "Plan a 5-day trip to Paris focusing on culture and food",
    "Create a 3-day itinerary for Bali with beach activities and temples",

    # 8. Combined Scenarios
    "I'm traveling to London next week. First, check the weather, then find hotels near central London, and suggest some restaurants",
    "I need to fly from Dubai to Singapore on March 10th. Also check the currency exchange rate and plan a 4-day itinerary"
]

for idx, scenario in enumerate(scenarios, 1):
    print(f"\033[1mQuery {idx}:\033[0m {scenario}\n")
    final_response = None
    for event in app.stream({"messages": [("user", scenario)]}):
        for node_name, value in event.items():
            if node_name == "tools":
                for tool_msg in value["messages"]:
                    print(f"\033[1mUsing Tool:\033[0m {tool_msg.name}")

            if "messages" in value:
                final_response = value["messages"][-1]

    if final_response:
        chatbot_response = final_response.content
        print(f"\033[1mTravel-Bot:\033[0m\n{chatbot_response}\n" + "="*100 + "\n")
    else:
        print("\033[1mTravel-Bot:\033[0m No response generated.\n" + "="*100 + "\n")


Query 1: Find flights from New York to London on March 15th, 2026

Using Tool: flight_search
Travel-Bot:
Here are some flight options from New York to London on March 15th, 2026:

1. **Airline:** AC  
   **Departure:** 16:50  
   **Duration:** 4 hours 31 minutes  
   **Price:** €208.43  
   **Stops:** 1  
   **Airport:** JFK to YXU

2. **Airline:** TP  
   **Departure:** 23:00  
   **Duration:** 12 hours 45 minutes  
   **Price:** €177.04  
   **Stops:** 1  
   **Airport:** JFK to LGW

3. **Airline:** VS  
   **Departure:** 09:20  
   **Duration:** 6 hours 50 minutes  
   **Price:** €199.18  
   **Stops:** 0  
   **Airport:** JFK to LHR

4. **Airline:** A1  
   **Departure:** 19:05  
   **Duration:** 6 hours 55 minutes  
   **Price:** €186.34  
   **Stops:** 0  
   **Airport:** JFK to LGW

5. **Airline:** AZ  
   **Departure:** 17:45  
   **Duration:** 13 hours 10 minutes  
   **Price:** €238.64  
   **Stops:** 1  
   **Airport:** JFK to LCY

If you need more options or assistance with